# Create Multimap Sequential Dataset

This notebook iterates through all 14 of the raw `.npz` datasets, processes them into sequences grouped by episode, and saves the final list of all episodes as a single `.pkl` file.

This is the recommended format for training an LSTM, as it preserves the temporal sequence of each episode while handling variable lengths.



In [1]:
import numpy as np
from pathlib import Path
import pickle
import os

# 1. Locate project root and the selected dataset run directory
# Try to get working directory, with fallback to notebook location if cwd is unavailable
try:
    project_root = Path.cwd()
except (FileNotFoundError, RuntimeError):
    # Fallback: use notebook's parent directory as reference
    # In Jupyter, __file__ may not work, so use a relative path approach
    project_root = Path(__file__).resolve().parent if '__file__' in dir() else Path.home()

# Walk up the directory tree to find mpc_datasets
for _ in range(10):
    if (project_root / "mpc_datasets").exists():
        break
    project_root = project_root.parent
else:
    raise FileNotFoundError(f"Could not find mpc_datasets ancestor. Looked from: {project_root}")

selected_run_dir = project_root / "mpc_datasets" / "run_20260420T215212Z"
if not selected_run_dir.exists():
    raise FileNotFoundError(f"Selected run directory does not exist: {selected_run_dir}")
print(f"Project root: {project_root}")
print(f"Selected dataset run: {selected_run_dir}")

# 2. Find all ST-MPC dataset files under the selected run
npz_paths = sorted(selected_run_dir.rglob("stmpc_*.npz"))
if not npz_paths:
    raise FileNotFoundError(f"No stmpc_*.npz files found under {selected_run_dir}")
print(f"Found {len(npz_paths)} ST-MPC dataset files")

# 3. Aggregate all episodes across all datasets
all_episode_sequences = []

total_steps = 0
for npz_path in npz_paths:
    map_name = npz_path.stem.replace("stmpc_", "")

    data = np.load(npz_path)

    if "observations" not in data.files or "lidar_scans" not in data.files:
        print(f"[WARN] Skipping {npz_path}: missing required arrays")
        continue

    # Drop x,y from observations: keep [delta, linear_vel_x, pose_theta]
    observations = data["observations"][:, 2:]  # (N, 3)
    lidar_scans = data["lidar_scans"]          # (N, 60)

    path_curvature_lookahead = data["path_curvature_lookahead"] if "path_curvature_lookahead" in data.files else None
    path_kappa_current = data["path_kappa_current"] if "path_kappa_current" in data.files else None
    path_centerline_idx_anchor = data["path_centerline_idx_anchor"] if "path_centerline_idx_anchor" in data.files else None
    path_s_anchor = data["path_s_anchor"] if "path_s_anchor" in data.files else None

    if path_curvature_lookahead is None:
        print(f"[WARN] Skipping {npz_path}: missing path_curvature_lookahead")
        continue

    path_curvature_lookahead = path_curvature_lookahead.astype(np.float32, copy=False)
    if path_curvature_lookahead.ndim != 2 or path_curvature_lookahead.shape[1] != 5:
        raise ValueError(f"Expected path_curvature_lookahead shape (N,5), got {path_curvature_lookahead.shape} in {npz_path}")
    inputs = np.concatenate([observations, lidar_scans, path_curvature_lookahead], axis=1)

    # Use expert_actions (clean MPC actions) as targets
    targets = data["expert_actions"] if "expert_actions" in data.files else None
    if targets is None:
        print(f"[WARN] Skipping {npz_path}: missing expert_actions")
        continue

    episode_ids = data["episode_ids"]
    unique_eps = np.unique(episode_ids)

    collision_flags = data["collision_flags"] if "collision_flags" in data.files else None
    boundary_flags = data["boundary_flags"] if "boundary_flags" in data.files else None

    for ep_id in unique_eps:
        mask = episode_ids == ep_id
        ep_inputs = inputs[mask]
        ep_targets = targets[mask]
        ep_path_curvature_lookahead = path_curvature_lookahead[mask] if path_curvature_lookahead is not None else None
        ep_path_kappa_current = path_kappa_current[mask] if path_kappa_current is not None else None
        ep_path_centerline_idx_anchor = path_centerline_idx_anchor[mask] if path_centerline_idx_anchor is not None else None
        ep_path_s_anchor = path_s_anchor[mask] if path_s_anchor is not None else None

        # Episode-level collision/boundary indicators
        has_collision = False
        has_boundary = False
        if collision_flags is not None:
            coll_ep = collision_flags[mask]
            has_collision = bool(coll_ep.any()) if coll_ep.size > 0 else False
        if boundary_flags is not None:
            bound_ep = boundary_flags[mask]
            has_boundary = bool(bound_ep.any()) if bound_ep.size > 0 else False

        all_episode_sequences.append({
            "inputs": ep_inputs,
            "targets": ep_targets,
            "path_curvature_lookahead": ep_path_curvature_lookahead,
            "path_kappa_current": ep_path_kappa_current,
            "path_centerline_idx_anchor": ep_path_centerline_idx_anchor,
            "path_s_anchor": ep_path_s_anchor,
            "map": map_name,
            "episode_id_local": int(ep_id),
            "has_collision": has_collision,
            "has_boundary": has_boundary,
        })

        total_steps += ep_inputs.shape[0]

    print(f"{map_name}: {len(unique_eps)} episodes, {inputs.shape[0]} steps")

print(f"\nTotal episodes aggregated (including collisions): {len(all_episode_sequences)}")
print(f"Total steps aggregated: {total_steps}")


Project root: /home/devin_work/work/f1tenth/ApproxiMPC
Selected dataset run: /home/devin_work/work/f1tenth/ApproxiMPC/mpc_datasets/run_20260420T215212Z
Found 47 ST-MPC dataset files
Austin_normal: 15 episodes, 239237 steps
Austin_reverse: 15 episodes, 238696 steps
BrandsHatch_normal: 15 episodes, 197175 steps
BrandsHatch_reverse: 15 episodes, 198263 steps
Budapest_normal: 15 episodes, 223289 steps
Budapest_reverse: 15 episodes, 225504 steps
Catalunya_normal: 15 episodes, 234404 steps
Catalunya_reverse: 15 episodes, 233601 steps
Drift2_normal: 15 episodes, 30051 steps
Drift2_reverse: 15 episodes, 28740 steps
Drift2_mirror_normal: 15 episodes, 15 steps
Drift2_mirror_reverse: 15 episodes, 29482 steps
Hockenheim_normal: 15 episodes, 200172 steps
Hockenheim_reverse: 15 episodes, 202076 steps
IMS_normal: 15 episodes, 162115 steps
IMS_reverse: 15 episodes, 162113 steps
Melbourne_normal: 15 episodes, 265311 steps
Melbourne_reverse: 15 episodes, 265500 steps
MexicoCity_normal: 15 episodes, 2007

In [2]:
# Helper: inspect episode length distribution before deciding on padding

# Option: drop collision episodes for this analysis as well
non_collision_episodes = [ep for ep in all_episode_sequences if not ep.get("has_collision", False)]

if len(all_episode_sequences) == 0:
    print("No episodes found. Run the aggregation cell above first.")
else:
    print(f"Total episodes (raw):        {len(all_episode_sequences)}")
    print(f"Episodes without collisions: {len(non_collision_episodes)}")
    print(f"Episodes with collisions:    {len(all_episode_sequences) - len(non_collision_episodes)}")

    if len(non_collision_episodes) == 0:
        print("No collision-free episodes to analyze.")
    else:
        # Compute lengths (number of steps) for each collision-free episode
        episode_lengths = [ep["inputs"].shape[0] for ep in non_collision_episodes]
        lengths_arr = np.asarray(episode_lengths)
        print("\nLength stats for collision-free episodes only:")
        print(f"  Min steps per episode:    {lengths_arr.min()}")
        print(f"  Max steps per episode:    {lengths_arr.max()}")
        print(f"  Mean steps per episode:   {lengths_arr.mean():.1f}")
        print(f"  Median steps per episode: {np.median(lengths_arr):.1f}")

        # Optional: rough histogram of lengths
        bins = [0, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000]
        hist, edges = np.histogram(lengths_arr, bins=bins)
        print("\nEpisode length histogram (steps, collision-free only):")
        for count, left, right in zip(hist, edges[:-1], edges[1:]):
            print(f"  [{left:4d}, {right:4d}): {count} episodes")


Total episodes (raw):        681
Episodes without collisions: 681
Episodes with collisions:    0

Length stats for collision-free episodes only:
  Min steps per episode:    1
  Max steps per episode:    21351
  Mean steps per episode:   13928.4
  Median steps per episode: 15092.0

Episode length histogram (steps, collision-free only):
  [   0, 6000): 60 episodes
  [6000, 7000): 0 episodes
  [7000, 8000): 0 episodes
  [8000, 9000): 0 episodes
  [9000, 10000): 28 episodes
  [10000, 11000): 37 episodes
  [11000, 12000): 13 episodes
  [12000, 13000): 78 episodes
  [13000, 14000): 90 episodes
  [14000, 15000): 25 episodes


In [3]:
# Helper: inspect why very short episodes occur (collisions, boundaries, etc.)

# Configure how many of the shortest episodes to inspect
num_short_episodes_to_show = 10

if len(all_episode_sequences) == 0:
    print("No episodes found. Run the aggregation cell first.")
else:
    # 1. Build a list of (idx, length, map, episode_id_local)
    epi_info = [
        (idx, ep["inputs"].shape[0], ep["map"], ep["episode_id_local"])
        for idx, ep in enumerate(all_episode_sequences)
    ]

    # 2. Sort by length ascending
    epi_info_sorted = sorted(epi_info, key=lambda x: x[1])

    # 3. Take the shortest few
    to_inspect = epi_info_sorted[:num_short_episodes_to_show]

    print(f"Inspecting the {len(to_inspect)} shortest episodes:")

    for global_idx, length, map_name, ep_id in to_inspect:
        ep = all_episode_sequences[global_idx]
        dataset_path = ep.get("dataset_path")
        if dataset_path is None:
            npz_path = next((p for p in selected_run_dir.rglob(f"stmpc_{map_name}.npz")), None)
            if npz_path is None:
                print(f"\nEpisode {global_idx} ({map_name}, ep_id={ep_id}, len={length}): cannot resolve dataset path")
                continue
        else:
            npz_path = Path(dataset_path)

        if not npz_path.exists():
            print(f"\nEpisode {global_idx} ({map_name}, ep_id={ep_id}, len={length}): data file missing at {npz_path}")
            continue

        data = np.load(npz_path)
        episode_ids = data["episode_ids"]
        mask = (episode_ids == ep_id)

        # Pull per-step flags for just this episode
        collision = data["collision_flags"][mask]
        boundary = data["boundary_flags"][mask]
        term = data["terminations"][mask]
        trunc = data["truncations"][mask]

        any_collision = bool(collision.any()) if collision.size > 0 else False
        any_boundary = bool(boundary.any()) if boundary.size > 0 else False
        any_term = bool(term.any()) if term.size > 0 else False
        any_trunc = bool(trunc.any()) if trunc.size > 0 else False

        # Look at last step flags to guess end reason
        end_reason = "step_limit_or_unknown"
        if term.size > 0 or trunc.size > 0:
            last_term = bool(term[-1])
            last_trunc = bool(trunc[-1])
            last_collision = bool(collision[-1]) if collision.size > 0 else False
            last_boundary = bool(boundary[-1]) if boundary.size > 0 else False

            if last_term:
                if last_collision:
                    end_reason = "collision_terminated"
                elif last_boundary:
                    end_reason = "boundary_terminated"
                else:
                    end_reason = "env_terminated"
            elif last_trunc:
                end_reason = "env_truncated"

        print(f"\nEpisode {global_idx}: map={map_name}, local_ep_id={ep_id}, steps={length}")
        print(f"  any_collision: {any_collision}, any_boundary: {any_boundary}")
        print(f"  any_terminated: {any_term}, any_truncated: {any_trunc}")
        print(f"  inferred_end_reason: {end_reason}")

Inspecting the 10 shortest episodes:

Episode 150: map=Drift2_mirror_normal, local_ep_id=0, steps=1
  any_collision: False, any_boundary: False
  any_terminated: False, any_truncated: False
  inferred_end_reason: step_limit_or_unknown

Episode 151: map=Drift2_mirror_normal, local_ep_id=1, steps=1
  any_collision: False, any_boundary: False
  any_terminated: False, any_truncated: False
  inferred_end_reason: step_limit_or_unknown

Episode 152: map=Drift2_mirror_normal, local_ep_id=2, steps=1
  any_collision: False, any_boundary: False
  any_terminated: False, any_truncated: False
  inferred_end_reason: step_limit_or_unknown

Episode 153: map=Drift2_mirror_normal, local_ep_id=3, steps=1
  any_collision: False, any_boundary: False
  any_terminated: False, any_truncated: False
  inferred_end_reason: step_limit_or_unknown

Episode 154: map=Drift2_mirror_normal, local_ep_id=4, steps=1
  any_collision: False, any_boundary: False
  any_terminated: False, any_truncated: False
  inferred_end_rea

In [4]:
# 4. Save list-of-episodes structure as a pickle file
# For this first training set, we exclude any episodes that experienced a collision.

non_collision_episodes = [ep for ep in all_episode_sequences if not ep.get("has_collision", False)]

output_dir = project_root / "datasets"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "multimap_lstm_sequences_422.pkl"

# 4a. Review the episode schema before saving
if len(non_collision_episodes) == 0:
    print("No episodes to save. Run the aggregation cell first.")
else:
    sample_episode = non_collision_episodes[0]
    print("Episode schema keys:", list(sample_episode.keys()))
    for key, value in sample_episode.items():
        if isinstance(value, np.ndarray):
            print(f"- {key}: ndarray shape={value.shape}, dtype={value.dtype}")
        else:
            print(f"- {key}: {type(value).__name__}")



Episode schema keys: ['inputs', 'targets', 'path_curvature_lookahead', 'path_kappa_current', 'path_centerline_idx_anchor', 'path_s_anchor', 'map', 'episode_id_local', 'has_collision', 'has_boundary']
- inputs: ndarray shape=(16409, 68), dtype=float32
- targets: ndarray shape=(16409, 2), dtype=float32
- path_curvature_lookahead: ndarray shape=(16409, 5), dtype=float32
- path_kappa_current: ndarray shape=(16409,), dtype=float32
- path_centerline_idx_anchor: ndarray shape=(16409,), dtype=int32
- path_s_anchor: ndarray shape=(16409,), dtype=float32
- map: str
- episode_id_local: int
- has_collision: bool
- has_boundary: bool


In [ ]:
# 4b. Save the dataset
with open(output_path, "wb") as f:
    pickle.dump(non_collision_episodes, f)

size_mb = os.path.getsize(output_path) / 1e6
print(f"Saved PKL dataset (no-collision episodes only) to: {output_path} ({size_mb:.2f} MB)")
print(f"Episodes saved: {len(non_collision_episodes)} of {len(all_episode_sequences)} total")

Saved PKL dataset (no-collision episodes only) to: /home/devin_work/work/f1tenth/ApproxiMPC/datasets/multimap_lstm_sequences_422.pkl (2769.89 MB)
Episodes saved: 681 of 681 total
